# mplstudio × scanpy

mplstudio detects the two most common scanpy plot styles and adds a **`palette=`** kwarg to carry scanpy's colour assignments directly into the GUI.

| Pattern | What scanpy does | What mplstudio does |
|---|---|---|
| Categorical UMAP | One `PathCollection` per category, no `ax.legend()` | Detects unlabeled collections, falls back to direct scan |
| Continuous UMAP | Single `PathCollection` + `set_array()`, label `""` | Picks up scalar-mapped collection, shows colormap controls |

**Install:**
```bash
pip install mplstudio scanpy
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scanpy as sc
import mplstudio

print(f"scanpy {sc.__version__}")

## 1. Categorical UMAP

Scanpy draws one `PathCollection` per category and no `ax.legend()` (it renders its own external legend via `AnchoredOffsetbox`).  
mplstudio's fallback scan picks up all labeled collections automatically.

In [ ]:
adata = sc.datasets.pbmc68k_reduced()
sc.pp.neighbors(adata)
sc.tl.umap(adata)

# palette dict — same structure as adata.uns["cell_type_colors"]
palette = dict(zip(
    adata.obs["bulk_labels"].cat.categories,
    adata.uns["bulk_labels_colors"],
))
print("Cell types:", list(palette.keys()))

In [ ]:
fig_cat, ax_cat = plt.subplots(figsize=(5, 4))
sc.pl.umap(adata, color="bulk_labels", ax=ax_cat, show=False)
plt.close()

# palette= kwarg → Colors section shows a "Custom" mode
mplstudio.studio(fig_cat, palette=palette)

**What to try in the panel:**

- **Colors → Custom**: restores the original scanpy palette at any time.
- **Colors → Manual**: individual pickers are pre-filled from the palette — tweak single categories.
- **Colors → Palette / Smart**: switch to a built-in palette or a perceptually-uniform auto palette.
- **Typography**: adjust font size and family for axis labels / title.
- **Axes**: tighten axis limits or add/remove grid.

## 2. Continuous UMAP (gene expression)

Scanpy plots gene expression as a single `PathCollection` with `set_array()` (per-cell values) and label `""`.  
mplstudio detects this as a *continuous* plot and shows the colormap picker.

In [ ]:
gene = adata.var_names[0]
fig_cont, ax_cont = plt.subplots(figsize=(5, 4))
sc.pl.umap(adata, color=gene, ax=ax_cont, show=False)
plt.close()

mplstudio.studio(fig_cont)

**What to try:**

- **Colors → Sequential**: swap `viridis` for `plasma`, `inferno`, or `magma`.
- **Colors → Diverging**: try `RdBu_r` or `coolwarm` to highlight cells above / below the mean.
- **Save**: export a publication-ready PNG at 300 dpi.

## 3. Heatmap (`sc.pl.heatmap`)

`sc.pl.heatmap` renders a gene × group matrix as `imshow` — a continuous artist.  
mplstudio detects it automatically and shows the **colormap picker** with a gradient preview.

In [ ]:
marker_genes = {
    "CD14+ Monocyte":    ["LYZ", "CST3", "CD14"],
    "CD19+ B":           ["CD79A", "MS4A1"],
    "CD4+/CD25 T Reg":  ["IL2RA", "FOXP3"],
    "CD4+/CD45RA+/CD25- Naive T": ["CCR7", "SELL"],
    "NK cell":           ["GNLY", "NKG7"],
}
# Only use genes present in adata
valid = {ct: [g for g in gs if g in adata.var_names]
         for ct, gs in marker_genes.items()}
plot_genes = [g for gs in valid.values() for g in gs][:12]

axes_dict = sc.pl.heatmap(
    adata, var_names=plot_genes, groupby="bulk_labels", show=False
)
fig_hm = axes_dict["heatmap_ax"].get_figure()
plt.close()

mplstudio.studio(fig_hm)

**What to try:**

- **Colors → Sequential**: `Reds` → `YlOrRd`, `viridis`, `plasma` 등으로 바꿔보기.
- **Colors → Diverging**: `RdBu_r`나 `coolwarm`으로 up/down-regulated 유전자 강조.
- **Typography**: 폰트 크기를 키우면 유전자 이름과 cell type 레이블 가독성이 올라감.
- **Figure size**: 유전자 수에 맞게 세로 길이 조정.

## 4. Violin plot (`sc.pl.violin`)

`sc.pl.violin`은 cell type별 유전자 발현 분포를 보여주는 그래프.  
Violin 패치(`PolyCollection`)는 레이블이 없기 때문에 **Colors 섹션은 "No labeled series"** 로 표시되며 색 변경은 지원되지 않는다.  
대신 **Typography, Figure size, Grid, Spines** 컨트롤이 논문용 다듬기에 유용하다.

In [ ]:
violin_genes = [g for g in ["LYZ", "NKG7", "CD79A"] if g in adata.var_names]

fig_vn, axes_vn = plt.subplots(1, len(violin_genes), figsize=(9, 3), sharey=False)
for ax_v, gene in zip(np.atleast_1d(axes_vn), violin_genes):
    sc.pl.violin(adata, keys=gene, groupby="bulk_labels",
                 ax=ax_v, show=False, rotation=35)
fig_vn.suptitle("Marker gene expression by cell type", y=1.02)
plt.tight_layout()
plt.close()

mplstudio.studio(fig_vn)

**What to try:**

- **Typography**: 폰트 크기를 높이면 유전자 이름(subplot title)과 cell type x-tick 레이블 가독성이 올라감.
- **Grid → On**: 수평 grid line을 켜면 발현 수치를 읽기 쉬워짐.
- **Spines → 2-Side** 또는 **None**: 논문 스타일에 맞게 spine을 정리.
- **Figure size**: 유전자를 더 추가할 때 가로 길이를 늘려 서브플롯 간격 확보.